In [1]:
import numpy as np
import pandas as pd
import os
import warnings
from PIL import Image
from tqdm import tqdm
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import timm
from multiprocessing import Pool
import cv2



/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [2]:
import shutil

try:
    shutil.rmtree("/kaggle/working/train")
    model_file_to_delete = "/kaggle/working/models"
    if os.path.isfile(model_file_to_delete):
        os.remove(model_file_to_delete)
except Exception:
    print("No such directories")




No such directories


In [3]:
def load_data(data_dir, split):
    csv_path = os.path.join(data_dir, f"{split}.csv")
    df = pd.read_csv(csv_path)
    img_dir = os.path.join(data_dir, f"{split}_images/")
    df["file_path"] = df["id_code"].map(lambda x: os.path.join(img_dir, f"{x}.png"))
    df["file_name"] = df["id_code"] + ".png"
    return df




In [4]:
data_dir = "/kaggle/input/aptos2019-blindness-detection/"
test_df = load_data(data_dir, "test")




In [5]:
def crop_img(img, percentage):
    img_arr = np.array(img)
    img_gray = cv2.cvtColor(img_arr, cv2.COLOR_BGR2GRAY)
    threshold = img_gray > 0.1 * np.mean(img_gray[img_gray != 0])
    row_sums = np.sum(threshold, axis=1)
    col_sums = np.sum(threshold, axis=0)
    rows = np.where(row_sums > img_arr.shape[1] * percentage)[0]
    cols = np.where(col_sums > img_arr.shape[0] * percentage)[0]
    min_row, min_col = np.min(rows), np.min(cols)
    max_row, max_col = np.max(rows), np.max(cols)
    crop_img = img_arr[min_row : max_row + 1, min_col : max_col + 1]
    return Image.fromarray(crop_img)




In [6]:
def resize_maintain_aspect(img, desired_size):
    old_width, old_height = img.size
    aspect_ratio = old_width / old_height
    if aspect_ratio > 1:
        new_width = desired_size
        new_height = int(desired_size / aspect_ratio)
    else:
        new_height = desired_size
        new_width = int(desired_size * aspect_ratio)
    resample_method = getattr(Image, "Resampling", Image).LANCZOS
    resized_img = img.resize((new_width, new_height), resample=resample_method)
    padded_image = Image.new("RGB", (desired_size, desired_size))
    x_offset = (desired_size - new_width) // 2
    y_offset = (desired_size - new_height) // 2
    padded_image.paste(resized_img, (x_offset, y_offset))
    return padded_image




In [7]:
def save_single(args):
    image_path, output_path_folder, percentage, output_size = args
    image = Image.open(image_path)
    croped_img = crop_img(image, percentage)
    image_resized = resize_maintain_aspect(croped_img, desired_size=output_size[0])
    output_file_path = os.path.join(output_path_folder, os.path.basename(image_path))
    image_resized.save(output_file_path)




In [8]:
def fast_image_resize(df, output_path_folder, percentage, output_size=None):
    """Uses multiprocessing to make it fast"""
    if not output_size:
        warnings.warn("Need to specify output_size! For example: output_size=100")
        return
    if not os.path.exists(output_path_folder):
        os.makedirs(output_path_folder)
    jobs = [
        (df.file_path.iloc[i], output_path_folder, percentage, output_size)
        for i in range(len(df))
    ]
    with Pool() as p:
        list(tqdm(p.imap_unordered(save_single, jobs), total=len(jobs)))




In [9]:
percentage = 0.01
fast_image_resize(
    test_df,
    "/kaggle/working/test/images_resized/",
    percentage,
    output_size=(100, 100),
)



100%|██████████| 367/367 [00:15<00:00, 24.42it/s]


In [10]:
# Load and preprocess training images for a short fine‑tuning phase
train_df = load_data(data_dir, "train")
fast_image_resize(
    train_df,
    "/kaggle/working/train/images_resized/",
    percentage,
    output_size=(100, 100),
)




100%|██████████| 3295/3295 [02:03<00:00, 26.73it/s]


In [11]:
class BlindnessDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None, test=False):
        self.annotations = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform
        self.test = test

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.annotations.iloc[idx, 0] + ".png")
        image = Image.open(img_name).convert("RGB")
        if self.transform:
            image = self.transform(image)
        if self.test:
            return image
        else:
            label = int(self.annotations.iloc[idx, 1])
            return image, label




In [12]:
transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)



In [13]:
test_csv_file = "/kaggle/input/aptos2019-blindness-detection/test.csv"
test_root_dir = "/kaggle/working/test/images_resized/"
test_dataset = BlindnessDataset(
    test_csv_file, test_root_dir, transform=transform, test=True
)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)



In [14]:
model_paths = {
    "inception_v4": "/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/3/inception_v4.pth",
}
model_names = {
    "inception_v4": "inception_v4",
}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
models_list = []
for model_key, path in model_paths.items():
    model_name = model_names[model_key]
    model = timm.create_model(model_name, pretrained=True, num_classes=5)
    try:
        state = torch.load(path, map_location=device)
        model.load_state_dict(state)
        print(f"Loaded checkpoint for {model_key}")
    except Exception as e:
        print(
            f"Could not load checkpoint for {model_key} ({e}); using ImageNet pretrained weights."
        )
    model.to(device)
    model.eval()
    models_list.append(model)



model.safetensors:   0%|          | 0.00/171M [00:00<?, ?B/s]

Could not load checkpoint for inception_v4 ([Errno 2] No such file or directory: '/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/3/inception_v4.pth'); using ImageNet pretrained weights.


In [15]:
# Simple weighting – only one model, so weight = 1.0
weights = {"inception_v4": 1.0}



In [16]:
# ------------------------------------------------------------------
# Fine‑tune the model on the provided training set (2 epochs)
# This modest training should noticeably improve predictions toward the target.
# ------------------------------------------------------------------
train_csv_file = "/kaggle/input/aptos2019-blindness-detection/train.csv"
train_root_dir = "/kaggle/working/train/images_resized/"
train_dataset = BlindnessDataset(
    train_csv_file, train_root_dir, transform=transform, test=False
)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(models_list[0].parameters(), lr=1e-3)

models_list[0].train()
for epoch in range(2):
    epoch_loss = 0.0
    for imgs, lbls in tqdm(train_loader, desc=f"Training epoch {epoch+1}"):
        imgs = imgs.to(device)
        lbls = lbls.to(device)
        optimizer.zero_grad()
        logits = models_list[0](imgs)
        loss = criterion(logits, lbls)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch+1} average loss: {avg_loss:.4f}")

models_list[0].eval()



Training epoch 1: 100%|██████████| 103/103 [00:13<00:00,  7.66it/s]


Epoch 1 average loss: 0.7744


Training epoch 2: 100%|██████████| 103/103 [00:12<00:00,  8.06it/s]

Epoch 2 average loss: 0.6286


InceptionV4(
  (features): Sequential(
    (0): ConvNormAct(
      (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), bias=False)
      (bn): BatchNormAct2d(
        32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
        (drop): Identity()
        (act): ReLU(inplace=True)
      )
    )
    (1): ConvNormAct(
      (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), bias=False)
      (bn): BatchNormAct2d(
        32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
        (drop): Identity()
        (act): ReLU(inplace=True)
      )
    )
    (2): ConvNormAct(
      (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn): BatchNormAct2d(
        64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
        (drop): Identity()
        (act): ReLU(inplace=True)
      )
    )
    (3): Mixed3a(
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
      (c

In [17]:
all_outputs = []
with torch.no_grad():
    for images in tqdm(test_loader, desc="Inference"):
        images = images.to(device)
        outputs = []
        for model_key, model in zip(model_paths.keys(), models_list):
            logits = model(images)
            probs = nn.functional.softmax(logits, dim=1)
            weighted = weights.get(model_key, 0) * probs
            outputs.append(weighted.unsqueeze(0))
        if outputs:
            stacked = torch.cat(outputs)  # (n_models, batch, classes)
            weighted_sum = torch.sum(stacked, dim=0)  # (batch, classes)
            all_outputs.extend(weighted_sum.cpu().numpy())

all_outputs = np.array(all_outputs)
final_predictions = np.argmax(all_outputs, axis=1)



Inference: 100%|██████████| 23/23 [00:00<00:00, 29.91it/s]


In [18]:
submission_df = pd.DataFrame(
    {"id_code": pd.read_csv(test_csv_file)["id_code"], "diagnosis": final_predictions}
)
submission_path = "submission.csv"
submission_df.to_csv(submission_path, index=False)
print(f"Submission saved to {submission_path}")

Submission saved to submission.csv
